In [ ]:
# ---------------
# Dependencies
# ---------------

import torch
import random
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

import pandas
import numpy
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    classification_report,
    average_precision_score,
)

In [ ]:
# ------------------
# Reproducibility
# ------------------


def set_seed(seed: int):
    random.seed(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


seed = 50
set_seed(seed)

In [ ]:
# ------------------------------------
# Device Setup (Is CUDA available?)
# ------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ", DEVICE)

In [ ]:
# -----------------------------------
# Load Dataset from GDrive (Colab)
# -----------------------------------

from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ---------------
# Load dataset
# ---------------

df = pandas.read_csv("/content/dataset.csv")

if "hash" in df.columns:
    df = df.drop(columns=["hash"])

In [ ]:
# -------------
# Data Setup
# -------------

X = df.drop(columns=["malware"]).values.astype(numpy.float32)
y = df["malware"].values.astype(numpy.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.30, stratify=df["malware"], random_state=42
)

X_train = torch.tensor(X_train).to(DEVICE)
X_val = torch.tensor(X_val).to(DEVICE)

y_train = torch.tensor(y_train).to(DEVICE)
y_val = torch.tensor(y_val).to(DEVICE)

In [ ]:
train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=128,
)

In [ ]:
VOCAB_SIZE = 307
SEQ_LEN = 100
NUM_CLASSES = 2
LATENT_DIM = 128
EMB_DIM = 128
BATCH_SIZE = 128
EPOCHS = 100

In [ ]:
# Discriminator
class TransformerDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMB_DIM)
        self.pos_embedding = nn.Embedding(SEQ_LEN, EMB_DIM)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=EMB_DIM, nhead=4, dim_feedforward=256, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.fc = nn.Sequential(
            nn.Linear(EMB_DIM, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(128, NUM_CLASSES + 1),
        )

    def forward(self, x, embedded=False, return_features=False):
        if not embedded:
            x = x.long()
            out = self.embedding(x)
        else:
            out = x

        batch_size, seq_len, _ = out.size()
        positions = (
            torch.arange(0, seq_len, device=out.device)
            .unsqueeze(0)
            .expand(batch_size, seq_len)
        )

        out = out + self.pos_embedding(positions)
        features = self.transformer(out)
        features = features.mean(dim=1)
        logits = self.fc(features)

        if return_features:
            return logits, features

        return logits

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        self.init_fc = nn.Linear(LATENT_DIM, 256)
        self.rnn = nn.GRU(input_size=EMB_DIM, hidden_size=256, batch_first=True)
        self.token_proj = nn.Linear(256, VOCAB_SIZE)
        self.start_token = nn.Parameter(torch.zeros(1, 1, EMB_DIM))

    def forward(self, z, temperature=0.5):
        batch_size = z.size(0)
        h0 = torch.tanh(self.init_fc(z)).unsqueeze(0)
        inputs = self.start_token.repeat(batch_size, SEQ_LEN, 1)
        outputs, _ = self.rnn(inputs, h0)
        logits = self.token_proj(outputs)
        return F.gumbel_softmax(logits, tau=temperature, hard=True)

In [ ]:
D = TransformerDiscriminator().to(DEVICE)
G = Generator().to(DEVICE)

optimizer_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizer_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))

In [ ]:
for epoch in range(EPOCHS):

    D.train()
    G.train()

    for real_x, real_y in train_loader:

        real_x = real_x.to(DEVICE)
        real_y = real_y.to(DEVICE)
        batch_size = real_x.size(0)

        """Discriminator Training"""
        optimizer_D.zero_grad()

        # Real
        logits_real = D(real_x)
        loss_real = F.cross_entropy(logits_real, real_y)

        # Fake
        z = torch.randn(batch_size, LATENT_DIM).to(DEVICE)
        fake_probs = G(z)

        # Soft tokens to embedding space
        fake_emb = torch.matmul(fake_probs, D.embedding.weight)

        logits_fake = D(fake_emb, embedded=True)

        fake_labels = torch.full((batch_size,), NUM_CLASSES, device=DEVICE)

        loss_fake = F.cross_entropy(logits_fake, fake_labels)

        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizer_D.step()

        """Generator Training"""
        optimizer_G.zero_grad()

        z = torch.randn(batch_size, LATENT_DIM).to(DEVICE)
        fake_probs = G(z)

        fake_emb = torch.matmul(fake_probs, D.embedding.weight)

        # Get features from discriminator
        logits_fake, feat_fake = D(fake_emb, embedded=True, return_features=True)
        _, feat_real = D(real_x, return_features=True)

        # Feature matching loss
        loss_G = F.mse_loss(feat_fake.mean(dim=0), feat_real.mean(dim=0))

        loss_G.backward()
        optimizer_G.step()

    print(f"Epoch {epoch+1} | D {loss_D:.4f} | G {loss_G:.4f}")

In [ ]:
torch.save(D.state_dict(), f"transformer_sgan_seed_{seed}.pt")

In [ ]:
D.load_state_dict(torch.load(f"transformer_sgan_seed_{seed}.pt"))
D.eval()

with torch.no_grad():
    logits = D(X_val)
    probs = F.softmax(logits[:, :NUM_CLASSES], dim=1)
    preds = torch.argmax(probs, dim=1)

print(classification_report(y_val.cpu().numpy(), preds.cpu().numpy(), digits=4))

print(
    "PR-AUC:", average_precision_score(y_val.cpu().numpy(), probs[:, 1].cpu().numpy())
)

print("ROC-AUC:", roc_auc_score(y_val.cpu().numpy(), probs[:, 1].cpu().numpy()))

52m

seed 10
```
             precision    recall  f1-score   support

           0     0.8561    0.7160    0.7798       324
           1     0.9929    0.9970    0.9949     12839

    accuracy                         0.9900     13163
   macro avg     0.9245    0.8565    0.8874     13163
weighted avg     0.9895    0.9900    0.9896     13163

PR-AUC: 0.998464535291302
ROC-AUC: 0.965382529503567
```

seed 20
```
              precision    recall  f1-score   support

           0     0.9157    0.7037    0.7958       324
           1     0.9926    0.9984    0.9955     12839

    accuracy                         0.9911     13163
   macro avg     0.9541    0.8510    0.8956     13163
weighted avg     0.9907    0.9911    0.9905     13163

PR-AUC: 0.9979657789092905
ROC-AUC: 0.9543968800693104
```

seed 30

```
              precision    recall  f1-score   support

           0     0.8769    0.7253    0.7939       324
           1     0.9931    0.9974    0.9953     12839

    accuracy                         0.9907     13163
   macro avg     0.9350    0.8614    0.8946     13163
weighted avg     0.9902    0.9907    0.9903     13163

PR-AUC: 0.9986431775461622
ROC-AUC: 0.9659624802516252
```

seed 40

```
              precision    recall  f1-score   support

           0     0.8684    0.7130    0.7831       324
           1     0.9928    0.9973    0.9950     12839

    accuracy                         0.9903     13163
   macro avg     0.9306    0.8551    0.8890     13163
weighted avg     0.9897    0.9903    0.9898     13163

PR-AUC: 0.9992361017208466
ROC-AUC: 0.9769279365821153
```